# Notebook 4 · Text to features

Six short descriptions stand in for documents from a cognitive-science corpus. The
notebook follows the path from text to a document–term matrix, TF–IDF vectors, and a
three-dimensional token representation.

In [ ]:
from pathlib import Path

class Check:
    def _result(self, passed, success, hint):
        if passed:
            print(f"✅ {success}")
        else:
            print("✗ Not correct. Open the hint if needed.")
        return passed

    def equal(self, actual, expected, success="Correct.", hint="Value does not match the expected result."):
        try:
            passed = actual == expected
            if hasattr(passed, "all"):
                passed = bool(passed.all())
        except Exception:
            passed = False
        return self._result(bool(passed), success, hint)

    def shape(self, actual, expected, success="Shape is correct.", hint="Shape is incorrect."):
        return self._result(tuple(actual.shape) == tuple(expected), success, hint)

    def columns(self, frame, expected, success="Columns are correct.", hint="Inspect frame.columns and select with a list of names."):
        return self._result(list(frame.columns) == list(expected), success, hint)

    def choice(self, actual, expected, explanations):
        normalised = str(actual).strip().upper()
        hint = explanations.get(normalised, "Choose one of the listed letters.")
        return self._result(normalised == expected.upper(), "Correct.", hint)

check = Check()

In [ ]:
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

documents = [
    "attention selects relevant visual information",
    "visual attention changes reaction time",
    "memory retrieval depends on context",
    "working memory supports language comprehension",
    "language models learn contextual representations",
    "reaction time measures lexical processing",
]

## 1 · Build a document–term matrix

Fit a `CountVectorizer` to `documents`. Store the sparse matrix in `counts` and the
feature names in `terms`.

In [ ]:
vectorizer = ...
counts = ...  # transform the documents
terms = ...

<details class="notebook-hint">
<summary><strong>Hint</strong></summary>

Create `CountVectorizer()`, call `.fit_transform(documents)`, then call `.get_feature_names_out()` on the fitted vectorizer.

</details>

In [ ]:
check.shape(counts, (6, 24))
check.equal(len(terms), 24)
check.equal("attention" in terms, True)

<details class="notebook-answer">
<summary><strong>Reveal answer</strong></summary>

```python
vectorizer = CountVectorizer()
counts = vectorizer.fit_transform(documents)
terms = vectorizer.get_feature_names_out()
```

</details>

## 2 · Read the matrix

Calculate the number of counted tokens in each document. Then find the column index
for `reaction` and extract that column as a one-dimensional array.

In [ ]:
document_lengths = ...
reaction_index = ...
reaction_counts = ...  # extract one term column

<details class="notebook-hint">
<summary><strong>Hint</strong></summary>

Sum `counts` across columns with `axis=1` and use `.A1` to obtain an array. Find a matching term with `list(terms).index(...)`; select that matrix column and use `.toarray().ravel()`.

</details>

In [ ]:
check.equal(document_lengths.tolist(), [5, 5, 5, 5, 5, 5])
check.equal(reaction_counts.tolist(), [0, 1, 0, 0, 0, 1])

<details class="notebook-answer">
<summary><strong>Reveal answer</strong></summary>

```python
document_lengths = counts.sum(axis=1).A1
reaction_index = list(terms).index("reaction")
reaction_counts = counts[:, reaction_index].toarray().ravel()
```

</details>

## 3 · TF–IDF and document similarity

Fit `TfidfVectorizer` and calculate the full document-by-document cosine-similarity
matrix.

In [ ]:
tfidf = ...
similarities = ...

<details class="notebook-hint">
<summary><strong>Hint</strong></summary>

Use `TfidfVectorizer().fit_transform(documents)`, then pass the resulting matrix twice to `cosine_similarity`.

</details>

In [ ]:
check.shape(tfidf, (6, 24))
check.shape(similarities, (6, 6))
check.equal(bool(np.allclose(np.diag(similarities), 1.0)), True)

<details class="notebook-answer">
<summary><strong>Reveal answer</strong></summary>

```python
tfidf = TfidfVectorizer().fit_transform(documents)
similarities = cosine_similarity(tfidf, tfidf)
```

The diagonal is 1 because each document is identical to itself. Off-diagonal values
increase when documents share terms, weighted by how informative those terms are in
this small corpus.

</details>

## 4 · Transfer the axis reasoning to token vectors

The toy array below has shape `documents × tokens × embedding features`. Average over
tokens to create one vector per document.

In [ ]:
rng = np.random.default_rng(7)
token_vectors = rng.normal(size=(6, 8, 4))
document_vectors = ...

<details class="notebook-hint">
<summary><strong>Hint</strong></summary>

Tokens are axis 1. Averaging that axis should leave `documents × embedding features`.

</details>

In [ ]:
check.shape(document_vectors, (6, 4))

<details class="notebook-answer">
<summary><strong>Reveal answer</strong></summary>

```python
document_vectors = token_vectors.mean(axis=1)
```

The token axis disappears, leaving one four-feature representation for each document.

</details>

### Reflection · What did the representation forget?

Compare the document–term matrix with the averaged token vectors. What information is
absent from both? What additional information does the averaging operation discard?

In [ ]:
reflection_nlp = """
Both representations omit ...
Averaging also removes ...
"""

<details class="notebook-reflection">
<summary><strong>Compare your reasoning</strong></summary>

A bag-of-words matrix omits word order and much syntax. Contextual token vectors may
encode order and context before aggregation, but a simple mean removes token position
and makes it impossible to recover which token contributed which feature. The right
representation depends on the linguistic question.

</details>